# 🎙️ VoiceBatch Studio v2.1.3 - [Fixed & Expressions]
इसमें [laugh], [sigh] के साथ GitHub सिंक को पूरी तरह फिक्स किया गया है।

In [ ]:
# @title 🔑 Step 1: GitHub Link (Safe Mode)
import os, shutil

GITHUB_USER = "" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
REPO_NAME = "" # @param {type:"string"}

if GITHUB_USER and GITHUB_TOKEN and REPO_NAME:
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    
    # अगर फोल्डर पहले से है तो उसे साफ़ करना (Error Fix)
    if os.path.exists(REPO_NAME):
        print("🔄 पुराना फोल्डर साफ़ किया जा रहा है...")
        shutil.rmtree(REPO_NAME)
    
    !git clone {REPO_URL}
    os.chdir(REPO_NAME)
    os.makedirs("outputs", exist_ok=True)
    
    print("⏳ लाइब्रेरी इंस्टॉल हो रही हैं...")
    !pip install -q gradio librosa soundfile coqui-tts
    print(f"✅ {REPO_NAME} तैयार है!")
else:
    print("⚠️ भाई, पहले टोकन और यूजरनेम भरें!")

In [ ]:
# @title 🚀 Step 2: app.py (Strict Emotion Control)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os, re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def master_engine(text, audio_sample, speed, pitch, lang):
    if not audio_sample: return None
    
    # [laugh], [sigh] के लिए टेक्स्ट तैयार करना
    text = text.replace('...', '. ')
    text = re.sub(r'([।?!,:;])', r' \1 ', text)
    
    out_file = 'outputs/v_batch_final.wav'
    
    tts.tts_to_file(
        text=text, 
        speaker_wav=audio_sample, 
        language=lang, 
        file_path=out_file,
        split_sentences=True
    )
    
    y, sr = librosa.load(out_file)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_file, y, sr)
    return out_file

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎙️ Master Voice Studio v2.1.3')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Script (Tags: [laugh], [sigh])', lines=7)
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en'], label='Lang', value='hi')
            spd = gr.Slider(0.8, 1.2, 1.0, label='Speed')
            ptc = gr.Slider(-3, 3, 0, label='Pitch')
            btn = gr.Button('Generate 🚀', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Result')

    btn.click(master_engine, [txt, smp, spd, ptc, lng], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप लोड हो गया है!")
!python app.py

In [ ]:
# @title ⬆️ Step 3: GitHub Push (सेव करें)
!git config --global user.email "user@example.com"
!git config --global user.name "{GITHUB_USER}"
!git add .
!git commit -m "Expression Audio Update"
!git push
print("✅ GitHub पर सफलतापूर्वक सेव हो गया!")